# Newton Harmonic Balance

Implementation of the Newton-enabled harmonic balance method from the Leadenham write-up for a periodic first-order system

$$\dot{x}=f(t,x)=f(t+T,x).$$

The approximation keeps $M$ harmonics:

$$\hat{x}(t)=a+A c(t)+B s(t),$$

where $c_m(t)=\cos(2\pi m t/T)$ and $s_m(t)=\sin(2\pi m t/T)$.

The residual is

$$r(t)=f(t,\hat{x}(t))-\dot{\hat{x}}(t).$$

Galerkin projection gives the nonlinear algebraic system

$$r_a=\frac{1}{T}\int_0^T f(t,\hat{x})\,dt=0,$$

$$R_{A,im}=\frac{1}{T}\int_0^T f_i(t,\hat{x})c_m(t)\,dt-\frac{m\omega}{2}B_{im}=0,$$

$$R_{B,im}=\frac{1}{T}\int_0^T f_i(t,\hat{x})s_m(t)\,dt+\frac{m\omega}{2}A_{im}=0,$$

with $\omega=2\pi/T$. The Jacobian is assembled from $G(t)=\partial f/\partial x\vert_{\hat{x}(t)}$ and solved by Newton iteration.

In [ ]:
import numpy as np
from numpy import pi
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp


In [ ]:
def harmonic_basis(t, T, M):
    """Return cosine and sine bases with shape (M, Nt)."""
    omega = 2*pi/T
    harmonics = np.arange(1, M + 1)[:, None]
    tau = t[None, :]
    return np.cos(harmonics*omega*tau), np.sin(harmonics*omega*tau)


def reconstruct_state(a, A, B, c, s):
    """Evaluate x_hat(t)=a+A c(t)+B s(t) on the sampled period."""
    return a[:, None] + A @ c + B @ s


def pack_coefficients(a, A, B):
    return np.concatenate([a.ravel(), A.ravel(order="F"), B.ravel(order="F")])


def unpack_coefficients(u, D, M):
    a = u[:D]
    A = u[D:D + D*M].reshape((D, M), order="F")
    B = u[D + D*M:].reshape((D, M), order="F")
    return a, A, B


def hb_residual_and_jacobian(f, dfdx, T, a, A, B, samples_per_harmonic=32):
    """Assemble the harmonic-balance residual and Newton Jacobian.

    Parameters
    ----------
    f : callable
        f(t, x) -> array with shape (D,).
    dfdx : callable
        dfdx(t, x) -> array with shape (D, D).
    T : float
        Desired period.
    a, A, B : arrays
        Fourier coefficients of x_hat(t).
    samples_per_harmonic : int
        Period samples per retained harmonic for quadrature.
    """
    D, M = A.shape
    omega = 2*pi/T
    Nt = int(2**np.ceil(np.log2(max(64, samples_per_harmonic*M))))
    t = np.linspace(0.0, T, Nt, endpoint=False)
    c, s = harmonic_basis(t, T, M)
    x = reconstruct_state(a, A, B, c, s)

    fs = np.empty((D, Nt))
    Gs = np.empty((D, D, Nt))
    for k, tk in enumerate(t):
        fs[:, k] = f(tk, x[:, k])
        Gs[:, :, k] = dfdx(tk, x[:, k])

    ra = fs.mean(axis=1)
    RA = fs @ c.T / Nt - A*0.0
    RB = fs @ s.T / Nt - B*0.0
    for m in range(M):
        factor = 0.5*(m + 1)*omega
        RA[:, m] -= factor*B[:, m]
        RB[:, m] += factor*A[:, m]

    v = pack_coefficients(ra, RA, RB)
    W = np.zeros((D*(2*M + 1), D*(2*M + 1)))

    eyeD = np.eye(D)
    for i in range(D):
        pa = i
        for j in range(D):
            qa = j
            gij = Gs[i, j, :]
            W[pa, qa] = gij.mean()
            for n in range(M):
                qA = j + D*(n + 1)
                qB = j + D*(M + n + 1)
                W[pa, qA] = np.mean(gij*c[n])
                W[pa, qB] = np.mean(gij*s[n])

    for m in range(M):
        factor_m = 0.5*(m + 1)*omega
        for i in range(D):
            pA = i + D*(m + 1)
            pB = i + D*(M + m + 1)
            for j in range(D):
                qa = j
                gij = Gs[i, j, :]
                W[pA, qa] = np.mean(gij*c[m])
                W[pB, qa] = np.mean(gij*s[m])
                for n in range(M):
                    qA = j + D*(n + 1)
                    qB = j + D*(M + n + 1)
                    W[pA, qA] = np.mean(gij*c[m]*c[n])
                    W[pA, qB] = np.mean(gij*c[m]*s[n])
                    W[pB, qA] = np.mean(gij*s[m]*c[n])
                    W[pB, qB] = np.mean(gij*s[m]*s[n])
                    if i == j and m == n:
                        W[pA, qB] -= factor_m
                        W[pB, qA] += factor_m

    return v, W, t, x, fs


def hb_newton(f, dfdx, T, a0, A0, B0, tol=1e-10, max_iter=50,
              samples_per_harmonic=32, damping=1.0, verbose=True):
    """Newton solve for periodic harmonic-balance coefficients."""
    a = np.array(a0, dtype=float).copy()
    A = np.array(A0, dtype=float).copy()
    B = np.array(B0, dtype=float).copy()
    D, M = A.shape
    history = []

    for it in range(1, max_iter + 1):
        v, W, t, x, fs = hb_residual_and_jacobian(
            f, dfdx, T, a, A, B, samples_per_harmonic=samples_per_harmonic
        )
        du = -np.linalg.pinv(W) @ v
        err = np.max(np.abs(du))
        res = np.linalg.norm(v, ord=2)
        history.append({"iter": it, "step_inf": err, "residual_2": res})
        if verbose:
            print(f"iter {it:02d}: ||res||_2={res:.3e}, ||du||_inf={err:.3e}")

        u = pack_coefficients(a, A, B) + damping*du
        a, A, B = unpack_coefficients(u, D, M)
        if err < tol:
            break
    else:
        raise RuntimeError("HB Newton solve did not converge; increase max_iter or improve the initial guess.")

    c, s = harmonic_basis(t, T, M)
    x = reconstruct_state(a, A, B, c, s)
    return {
        "a": a,
        "A": A,
        "B": B,
        "t": t,
        "x": x,
        "history": history,
        "x0": a + A.sum(axis=1),
    }


In [ ]:
# Duffing oscillator written as a first-order periodic system.
# x1 = displacement, x2 = velocity
omega0 = 1.0
zeta = 0.03
alpha = 1.0
F = 0.2
Omega = 0.8
T = 2*pi/Omega


def duffing_f(t, x):
    return np.array([
        x[1],
        F*np.cos(Omega*t) - 2*zeta*omega0*x[1] - omega0**2*x[0] - alpha*x[0]**3,
    ])


def duffing_dfdx(t, x):
    return np.array([
        [0.0, 1.0],
        [-omega0**2 - 3*alpha*x[0]**2, -2*zeta*omega0],
    ])


In [ ]:
M = 5
D = 2

# Linear forced response gives a useful initial guess for Newton.
den = (omega0**2 - Omega**2) + 1j*(2*zeta*omega0*Omega)
X = F/den

a0 = np.zeros(D)
A0 = np.zeros((D, M))
B0 = np.zeros((D, M))
A0[0, 0] = np.real(X)
B0[0, 0] = np.imag(X)
A0[1, 0] = Omega*np.imag(X)
B0[1, 0] = -Omega*np.real(X)

hb = hb_newton(
    duffing_f,
    duffing_dfdx,
    T,
    a0,
    A0,
    B0,
    tol=1e-11,
    max_iter=40,
    samples_per_harmonic=64,
)

print("x(0) from HB:", hb["x0"])
print("constant term a:", hb["a"])
print("cosine coefficients A:\n", hb["A"])
print("sine coefficients B:\n", hb["B"])


In [ ]:
# Check against direct time integration over many forcing periods.
n_periods = 120
samples_per_period = 300
t_eval = np.linspace(0.0, n_periods*T, n_periods*samples_per_period + 1)
sol = solve_ivp(
    duffing_f,
    (t_eval[0], t_eval[-1]),
    hb["x0"],
    t_eval=t_eval,
    rtol=1e-9,
    atol=1e-11,
)

t_last = sol.t[-samples_per_period:]
x_last = sol.y[0, -samples_per_period:]
phase_t = np.mod(t_last, T)
order = np.argsort(phase_t)

t_hb = hb["t"]
x_hb = hb["x"][0]
x_interp = np.interp(phase_t[order], t_hb, x_hb, period=T)
max_period_error = np.max(np.abs(x_last[order] - x_interp))
print(f"max displacement mismatch over final period: {max_period_error:.3e}")


In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(9, 6), constrained_layout=True)

ax[0].plot(hb["t"], hb["x"][0], label="HB displacement", lw=2)
ax[0].plot(phase_t[order], x_last[order], "--", label="time integration, final period", alpha=0.8)
ax[0].set_xlabel("phase time")
ax[0].set_ylabel("x")
ax[0].legend()
ax[0].grid(True, alpha=0.3)

iters = [h["iter"] for h in hb["history"]]
steps = [h["step_inf"] for h in hb["history"]]
resids = [h["residual_2"] for h in hb["history"]]
ax[1].semilogy(iters, resids, "o-", label="residual")
ax[1].semilogy(iters, steps, "s-", label="Newton step")
ax[1].set_xlabel("Newton iteration")
ax[1].grid(True, which="both", alpha=0.3)
ax[1].legend()

plt.show()
